In [ ]:
%%capture cap
%run ../src/desp-authentication.py

In [2]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]
access_token

'Token successfully written to /home/koenifra/.polytopeapirc'

In [ ]:
S3_KEY = ""
S3_SECRET = ""

scenarios_path = "destine-climate-dt/climate_dt_scenarios_sorted.csv"
points_folder = "destine-climate-dt/points/"

In [3]:
import pandas as pd
import concurrent.futures
import xarray as xr
import os
import s3fs
from typing import Literal

In [4]:
eodc_s3 = s3fs.S3FileSystem(
    key=S3_KEY,
    secret=S3_SECRET,
    client_kwargs={
        "endpoint_url": "https://objects.eodc.eu"
    })

In [ ]:

scenarios_path = "destine-climate-dt/climate_dt_scenarios_sorted.csv"
points_folder = "destine-climate-dt/points/"

with eodc_s3.open(scenarios_path, "r") as f:
        scenarios = pd.read_csv(f, sep=";")

print(scenarios)

    params      model level type experiment resolution temporal extent  \
0  167/228   IFS-NEMO        sfc       hist       high       1990-2001   
1      167       ICON        sfc       hist       high       1990-2019   
2  167/228   IFS-NEMO        sfc   SSP3-7.0       high       2020-2050   
3      167       ICON        sfc   SSP3-7.0       high       2020-2050   
4  167/228   IFS-NEMO        sfc       cont       high       1990-2007   
5  167/228  IFS-FESOM        sfc       cont       high       2017-2023   
6  167/228  IFS-FESOM        sfc       hist       high       2017-2023   
7  167/228  IFS-FESOM        sfc  Tplus2.0K       high       2017-2023   

        activity       typeOfSimulation  
0          CMIP6  Historical simulation  
1          CMIP6  Historical simulation  
2    ScenarioMIP      Future projection  
3    ScenarioMIP      Future projection  
4     HighResMIP     Control simulation  
5  story-nudging   Storyline simulation  
6  story-nudging   Storyline simulation

In [7]:
for _, scenario in scenarios.iloc[:1].iterrows():
    model = scenario["model"]
    params = scenario["params"]
    leveltype = scenario["level type"]
    experiment = scenario["experiment"]
    resolution = scenario["resolution"]
    temporalextent = scenario["temporal extent"]                
    activity = scenario["activity"]

    if model in ["IFS-NEMO", "ICON"]:
        points_path = f"{points_folder}points_{model}_{activity}.csv"
    else:  # IFS-FESOM
        points_path = f"{points_folder}points_{model}_{experiment}.csv"

    with eodc_s3.open(points_path, "r") as f:
        points = pd.read_csv(f)
    print(points_path)

    store_path = f"destine-climate-dt/Austria/{scenario['model']}_{scenario['level type']}_{scenario['activity']}_{scenario['experiment']}.zarr"
    print(f"Data will be stored in: {store_path}")

destine-climate-dt/points/points_IFS-NEMO_CMIP6.csv
Data will be stored in: destine-climate-dt/Austria/IFS-NEMO_sfc_CMIP6_hist.zarr


In [8]:
eodc_s3.exists("destine-climate-dt/Austria")

True

In [7]:
def extract_cdt_ts(experiment: str,
                   activity: str,
                   level_type: str,
                   datestring: str,
                   model: str,
                   parameter: str,
                   location: list,
                   feature: Literal["timeseries", "polygon"]="timeseries",
                   time_resolution: str="0000/to/2300",
                   resolution: str="high"):
    import earthkit.data
    
    if feature == "timeseries":
        feature_dict = {
            "type" : "timeseries",
            "points": location,
            "time_axis": "date"
        }
    elif feature == "polygon":
        feature_dict = {
            "type" : "polygon",
            "shape": location
        }
    else:
        raise TypeError("feature not supported")
    
    request = {
        # static parameters of climate dt data
        "class": "d1",
        "dataset": "climate-dt",
        "generation": "1",
        "expver": "0001",
        "stream": "clte",
        "type": "fc",
        # generic
        "activity": activity,
        "experiment": experiment,
        "levtype": level_type,
        "date": datestring,
        "model": model,
        "param": parameter,
        # "param": "167/228",
        # "param": "141",
        "realization": "1",
        "resolution": resolution,
        "time": time_resolution,
        "feature": feature_dict
    }

    # commented out to check if only one level is request it gets faster or not
    if level_type == "sol":
        # request["levelist"] = "1/to/5"
        request["levelist"] = "1"
    
    try:
        ds = earthkit.data.from_source("polytope", 
                                       "destination-earth", 
                                       request, stream=False, 
                                       address='polytope.lumi.apps.dte.destination-earth.eu')
        return ds.to_xarray()
    except Exception as e:
        print(e)
        return None

In [8]:
def get_datestring(temp_extent: str):
    '''
    Example output string "20200101/to/20210101"
    '''
    years = temp_extent.split("-")
    return f"{years[0]}0101/to/{years[1]}1231"
    

In [9]:
def get_station_data(station, scenario):
    latlon = [[float(station["latitude"]), float(station["longitude"])]]

    ts = extract_cdt_ts(
        experiment=scenario["experiment"],
        activity=scenario["activity"],
        level_type=scenario["level type"],
        datestring=get_datestring(scenario["temporal extent"]),
        model=scenario["model"],
        parameter=scenario["params"],
        location=latlon
    )

    if ts is None:
        return None

    point_id = int(station["points"])

    return ts.stack(pointid=("latitude", "longitude")) \
             .reset_index("pointid") \
             .assign_coords({"stationid": ("pointid", [station["points"]])}) \
             .assign_coords({"pointid": [point_id]}) \
             .transpose("pointid", ...)

In [ ]:
zarr_store = s3fs.S3Map(root=store_path, s3=eodc_s3)

with concurrent.futures.ThreadPoolExecutor(max_workers=25) as executor:
    futures = []
    for ind, station in points.iloc[:15].iterrows():
        futures.append(executor.submit(get_station_data, station=station, scenario=scenario))
    
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        if result is None:
            print("Skipping failed station")
            continue
        if not os.path.exists(zarr_store):
            result.chunk(chunks={
                "pointid": 1,
                "levelist": 1,
                "number": 1,
                "datetime": 1,
                "t": "auto"
            }).to_zarr(store=zarr_store, mode="w")
        else:
            result.chunk(chunks={
                "pointid": 1,
                "levelist": 1,
                "number": 1,
                "datetime": 1,
                "t": "auto"
            }).to_zarr(store=zarr_store, append_dim="pointid")

Exception ignored in: <function tqdm.__del__ at 0x7f1d1c394b80>
Traceback (most recent call last):
  File "/home/koenifra/Projects/dt-climate-zarr/dt/lib/python3.11/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/koenifra/Projects/dt-climate-zarr/dt/lib/python3.11/site-packages/tqdm/std.py", line 1277, in close
    if self.last_print_t < self.start_t + self.delay:
       ^^^^^^^^^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'last_print_t'
2026-05-05 10:00:12 - INFO - Key read from /home/koenifra/.polytopeapirc
2026-05-05 10:00:12 - INFO - Sending request...
{'request': 'activity: CMIP6\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            'date: 19900101/to/20011231\n'
            'experiment: hist\n'
            "expver: '0001'\n"
            'feature:\n'
            '  points:\n'
            '  - - 46.473481161792\n'
            '    - 14.177419354839\n'
            '  time_axis: date\n'
            '  type: timeserie

In [11]:
zarr_store


In [12]:
eodc_s3.exists(store_path)

False

2026-05-05 09:59:21 - INFO - The current status of the request is 'processing'
2026-05-05 09:59:25 - INFO - The current status of the request is 'processed'
2026-05-05 09:59:27 - INFO - The current status of the request is 'processing'                    
2026-05-05 09:59:29 - INFO - The current status of the request is 'processed'
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.11/logging/__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "/home/koenifra/Projects/dt-climate-zarr/dt/lib/python3.11/site-packages/ipykernel/iostream.py", line 760, in write
    self._schedule_flush()
  File "/home/koenifra/Projects/dt-climate-zarr/dt/lib/python3.11/site-packages/ipykernel/iostream.py", line 656, in _schedule_flush
    self.pub_thread.schedule(_schedule_in_thread)
  File "/home/koenifra/Projects/dt-climate-zarr/dt/lib/python3.11/site-packages/ipykernel/iostream.py", line 339, in schedule
    self._event_pipe.send(b"")
  File "/h

Socket operation on non-socket